# Experiment: Run HPT with GridSearch to build a Naive Bayes model

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
import string
import pandas as pd

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.pipeline import Pipeline
from sklearn.naive_bayes import MultinomialNB, BernoulliNB
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import f1_score

import nltk
from nltk.stem.snowball import SnowballStemmer
from nltk.tokenize import word_tokenize

from src import utils
import warnings


# Parameters

In [3]:
RND_SEED = 123
PCT_TEST = 0.2
K_FOLD = 3

EXPERIMENT = "exp01_hpt_nb"

# Paths
path_interim = os.path.join("data", "interim")
path_experiment =  os.path.join(path_interim, EXPERIMENT)

# Input
file_train = "train.csv"

# Output
file_exp = "df_exp_summary.csv"

In [4]:


utils.create_or_clean_folder(path_experiment)

Experiment folder already exists.
Removed: df_exp_summary.csv


# Load data

In [5]:
path_data_train = os.path.join(path_interim, file_train)

df_train = pd.read_csv(path_data_train)
df_train.head()

,x_text,y_is_nf
0,Poder crear cualquier mérito y asociarlo a mi ...,0
1,Como usuario autenticado quiero indicar que to...,0
2,Un usuario registrado o anónimo visualiza la l...,0
3,Como asesor quiero ver una lista de los experi...,0
4,Se podrá operar desde cualquier dispositivo a ...,1


# Build Pipeline

In [6]:
# Helper Cell: Tokenization and stemming in Spanish
import typing
import string



def tokenizer_stemmer_es(text) -> typing.List[str]:
 
    # drop puntuation and stopwords
    stopword_es = nltk.corpus.stopwords.words('spanish')
    # stemming
    stemmer = SnowballStemmer("spanish")

    clean_words = [word for word in word_tokenize(text) if word not in string.punctuation and word.lower() not in stopword_es] # list[str]
    return [stemmer.stem(word) for word in clean_words]  # list[str]


stopwords_es = nltk.corpus.stopwords.words('spanish')

example = df_train.loc[0, "x_text"]
ex_stem = tokenizer_stemmer_es(example)

print(f"{example=}")
print(f"{ex_stem=}")


example='Poder crear cualquier mérito y asociarlo a mi perfil de usuario.'
ex_stem=['pod', 'cre', 'cualqui', 'merit', 'asoci', 'perfil', 'usuari']


In [7]:
# Choose appropiate instances of XXXVectorizer and BernoulliXXX
tfbin_unigrams = CountVectorizer(
    strip_accents="ascii",
    lowercase=True,
    tokenizer=tokenizer_stemmer_es,
    ngram_range=(1, 1),
    binary=True,
)


clf_nbber = BernoulliNB()

# Create the pipeline
skl_pl = Pipeline([
    ('fte', tfbin_unigrams),
    ('clf', clf_nbber)
])


# Cross validate the model

Use the GridSearchCV object but with only a single configuration,
in order to maintain experiments scheme easily comparable.
You could also use other CV methods

Remember to use always the same number of CV Folds and the same CV metric on 
every experiment!


In [8]:
X_train = df_train['x_text']
y_train = df_train['y_is_nf']

# Change at will
param_grid = {
    # pipelinestep__parameter: [list of parameters values]
    # Check in documentation which parameters are worthly to trial
    # and what values do they expect
    'fte__max_features': [50, 100, 500, 1000],
    'fte__max_df': [0.1, 0.25, 0.5],
    'fte__min_df': [1, 3, 5],
}

grid_search = GridSearchCV(
    skl_pl,
    param_grid,
    cv=K_FOLD,  # maintain the same number of folds across the project
    scoring='f1',  # maintain the same scoring function (cv metric) across the project
    n_jobs=-1
    )

# Fit GridSearchCV on the training data
grid_search.fit(X_train, y_train)
print(f"{grid_search.best_score_:.4f}")

/home/estebanm/Escritorio/Master/pc4_nlp/.venv/lib/python3.12/site-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
/home/estebanm/Escritorio/Master/pc4_nlp/.venv/lib/python3.12/site-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
/home/estebanm/Escritorio/Master/pc4_nlp/.venv/lib/python3.12/site-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
/home/estebanm/Escritorio/Master/pc4_nlp/.venv/lib/python3.12/site-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
/home/estebanm/Escritorio/Master/pc4_nlp/.venv/lib/python3.12/site-packages/sklearn/feature_extraction/t

0.7477


In [9]:
df_exp_summary = pd.DataFrame(
    grid_search.cv_results_
)

df_exp_summary["experiment_id"] = EXPERIMENT
df_exp_summary 

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_fte__max_df,param_fte__max_features,param_fte__min_df,params,split0_test_score,split1_test_score,split2_test_score,mean_test_score,std_test_score,rank_test_score,experiment_id
0,0.140700,0.009601,0.061908,0.003038,0.10,50,1,"{'fte__max_df': 0.1, 'fte__max_features': 50, ...",0.542373,0.597015,0.411765,0.517051,0.077719,33,exp01_hpt_nb
1,0.143661,0.011689,0.061359,0.003896,0.10,50,3,"{'fte__max_df': 0.1, 'fte__max_features': 50, ...",0.571429,0.597015,0.411765,0.526736,0.081965,32,exp01_hpt_nb
2,0.147006,0.005293,0.063982,0.001175,0.10,50,5,"{'fte__max_df': 0.1, 'fte__max_features': 50, ...",0.524590,0.606061,0.363636,0.498096,0.100727,34,exp01_hpt_nb
3,0.139410,0.008553,0.061310,0.001214,0.10,100,1,"{'fte__max_df': 0.1, 'fte__max_features': 100,...",0.476190,0.655738,0.611111,0.581013,0.076327,29,exp01_hpt_nb
4,0.143297,0.014079,0.066037,0.006878,0.10,100,3,"{'fte__max_df': 0.1, 'fte__max_features': 100,...",0.595745,0.666667,0.611111,0.624507,0.030464,23,exp01_hpt_nb
5,0.146735,0.002423,0.064858,0.004946,0.10,100,5,"{'fte__max_df': 0.1, 'fte__max_features': 100,...",0.549020,0.642857,0.611111,0.600996,0.038971,28,exp01_hpt_nb
6,0.134390,0.012351,0.063383,0.002690,0.10,500,1,"{'fte__max_df': 0.1, 'fte__max_features': 500,...",0.437500,0.500000,0.333333,0.423611,0.068746,35,exp01_hpt_nb
7,0.118466,0.005351,0.060126,0.001975,0.10,500,3,"{'fte__max_df': 0.1, 'fte__max_features': 500,...",0.523810,0.755556,0.684211,0.654525,0.096911,21,exp01_hpt_nb
8,0.123620,0.000544,0.076620,0.017965,0.10,500,5,"{'fte__max_df': 0.1, 'fte__max_features': 500,...",0.565217,0.682927,0.594595,0.614246,0.050023,24,exp01_hpt_nb
9,0.115012,0.005119,0.059406,0.003233,0.10,1000,1,"{'fte__max_df': 0.1, 'fte__max_features': 1000...",0.451613,0.457143,0.344828,0.417861,0.051692,36,exp01_hpt_nb


# Diagnose the model

In [10]:
# Check DTM dimensions
skl_pl_fitted = grid_search.best_estimator_  
# Only one model is fit, as only one HPT configuration is passed

# Access the CountVectorizer part of the pipeline
skl_pl_fte = skl_pl_fitted.named_steps['fte']

# Get DTM with transform()
dtm_train = skl_pl_fte.transform(X_train)
print(f"{dtm_train.shape=}")  # columns: Number of terms in the vocabulary

dtm_train.shape=(311, 211)


In [11]:
# Check training predictions and scoring

y_hats_train = skl_pl_fitted.predict(X_train)  # get preds with predict()
f1_score_train = f1_score(
    y_true=y_train,
    y_pred=y_hats_train
)

print(f"{f1_score_train=}")  # Is comparable to CV metric?

f1_score_train=0.8129032258064516


# Write Experiments Results

In [12]:
df_exp_summary.to_csv(
    os.path.join(path_experiment, file_exp),
    index=False
)

# other experiments results and artifacts maybe useful

In [13]:
df_exp_summary

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_fte__max_df,param_fte__max_features,param_fte__min_df,params,split0_test_score,split1_test_score,split2_test_score,mean_test_score,std_test_score,rank_test_score,experiment_id
0,0.140700,0.009601,0.061908,0.003038,0.10,50,1,"{'fte__max_df': 0.1, 'fte__max_features': 50, ...",0.542373,0.597015,0.411765,0.517051,0.077719,33,exp01_hpt_nb
1,0.143661,0.011689,0.061359,0.003896,0.10,50,3,"{'fte__max_df': 0.1, 'fte__max_features': 50, ...",0.571429,0.597015,0.411765,0.526736,0.081965,32,exp01_hpt_nb
2,0.147006,0.005293,0.063982,0.001175,0.10,50,5,"{'fte__max_df': 0.1, 'fte__max_features': 50, ...",0.524590,0.606061,0.363636,0.498096,0.100727,34,exp01_hpt_nb
3,0.139410,0.008553,0.061310,0.001214,0.10,100,1,"{'fte__max_df': 0.1, 'fte__max_features': 100,...",0.476190,0.655738,0.611111,0.581013,0.076327,29,exp01_hpt_nb
4,0.143297,0.014079,0.066037,0.006878,0.10,100,3,"{'fte__max_df': 0.1, 'fte__max_features': 100,...",0.595745,0.666667,0.611111,0.624507,0.030464,23,exp01_hpt_nb
5,0.146735,0.002423,0.064858,0.004946,0.10,100,5,"{'fte__max_df': 0.1, 'fte__max_features': 100,...",0.549020,0.642857,0.611111,0.600996,0.038971,28,exp01_hpt_nb
6,0.134390,0.012351,0.063383,0.002690,0.10,500,1,"{'fte__max_df': 0.1, 'fte__max_features': 500,...",0.437500,0.500000,0.333333,0.423611,0.068746,35,exp01_hpt_nb
7,0.118466,0.005351,0.060126,0.001975,0.10,500,3,"{'fte__max_df': 0.1, 'fte__max_features': 500,...",0.523810,0.755556,0.684211,0.654525,0.096911,21,exp01_hpt_nb
8,0.123620,0.000544,0.076620,0.017965,0.10,500,5,"{'fte__max_df': 0.1, 'fte__max_features': 500,...",0.565217,0.682927,0.594595,0.614246,0.050023,24,exp01_hpt_nb
9,0.115012,0.005119,0.059406,0.003233,0.10,1000,1,"{'fte__max_df': 0.1, 'fte__max_features': 1000...",0.451613,0.457143,0.344828,0.417861,0.051692,36,exp01_hpt_nb
